# Week 3, Day 5: Reproducible research: notebooks, environments and Git branches

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w003/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
threshold = 0.5

In [ ]:
import numpy as np
rts = np.array([0.31, 0.62, 0.45, 0.88, 0.52])
print("slow trials:", (rts > threshold).sum())

### Exercise: Bug hunt: a function with a hidden input

`count_slow` quietly reads the global `threshold`, so its answer depends on which cells were run. Rewrite it to take the threshold as an argument, `count_slow(rts, threshold)`, and use only its arguments.

In [ ]:
def count_slow(rts):
    return int((rts > threshold).sum())

print(count_slow(rts))

In [ ]:
def test_argument():
    "count_slow uses the threshold it is given"
    import numpy as np
    x = np.array([0.2, 0.5, 0.9])
    assert count_slow(x, 0.4) == 2 and count_slow(x, 0.8) == 1

def test_no_global():
    "count_slow does not read a global threshold"
    import numpy as np
    g = count_slow.__globals__
    old = g.get("threshold")
    g["threshold"] = 100
    try:
        assert count_slow(np.array([0.2, 0.5, 0.9]), 0.4) == 2, "The result changed when the global changed."
    finally:
        g["threshold"] = old

test_argument()
test_no_global()
print("All checks passed")

In [ ]:
import os, sys, importlib
code = '''
"""Small tools for analysing decision-making sessions."""
import numpy as np

def proportion_right(choices):
    """Fraction of responses that were 'right' (choice == 1), ignoring no-response trials (0)."""
    choices = np.asarray(choices)
    valid = choices[choices != 0]
    return float((valid == 1).mean())
'''
os.makedirs("src", exist_ok=True)
open("src/behaviour.py", "w").write(code)
sys.path.insert(0, os.path.abspath("src"))
importlib.invalidate_caches()
import behaviour
print(behaviour.proportion_right([1, -1, 1, 0, 1]))
help(behaviour.proportion_right)

### Exercise: Add a function to the module

Append a function `psychometric(c, bias, sigma, lapse)` with a one-line docstring to `src/behaviour.py` (yesterday's formula), then reload the module and call it.

In [ ]:
extra = '''

def psychometric(c, bias, sigma, lapse):
    ...
'''
with open("src/behaviour.py", "a") as f:
    f.write(extra)
importlib.reload(behaviour)
print(behaviour.psychometric(0.1, 0.05, 0.15, 0.06))

In [ ]:
import importlib, numpy as np
def test_module_has_function():
    "behaviour.py defines psychometric"
    import behaviour
    importlib.reload(behaviour)
    assert hasattr(behaviour, "psychometric"), "Write psychometric into src/behaviour.py."

def test_function_works():
    "psychometric gives the right values"
    import behaviour
    importlib.reload(behaviour)
    assert abs(behaviour.psychometric(0.0, 0.0, 0.2, 0.0) - 0.5) < 1e-12
    assert abs(behaviour.psychometric(10.0, 0.0, 0.2, 0.1) - 0.9) < 1e-6

def test_docstring():
    "the function has a docstring"
    import behaviour
    importlib.reload(behaviour)
    assert behaviour.psychometric.__doc__, "Add a one-line docstring saying what it computes."

test_module_has_function()
test_function_works()
test_docstring()
print("All checks passed")

In [ ]:
import sys, platform, numpy, pandas, scipy, matplotlib
print("Python", sys.version.split()[0], "on", platform.system())
for m in [numpy, pandas, scipy, matplotlib]:
    print(f"{m.__name__}=={m.__version__}")

### Exercise: Save a result with its provenance

Write `save_result(path, value, seed, input_file)` that saves a JSON file holding `value`, `seed`, `input`, `versions` (a dictionary of the numpy, pandas and scipy versions) and `created` (the current date and time as text). Then save a bias of 0.08 from seed 1 and input `data/raw/session1.csv` to `results/bias_session1.json`.

In [ ]:
import json, os, datetime

def save_result(path, value, seed, input_file):
    record = {
        ...
    }
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(record, f, indent=2)

save_result("results/bias_session1.json", 0.08, 1, "data/raw/session1.csv")
print(open("results/bias_session1.json").read())

In [ ]:
import json, os
def test_file():
    "the result file exists and is valid JSON"
    assert os.path.exists("results/bias_session1.json")
    json.load(open("results/bias_session1.json"))

def test_contents():
    "it records the value, seed, input file and package versions"
    d = json.load(open("results/bias_session1.json"))
    for k in ["value", "seed", "input", "versions", "created"]:
        assert k in d, f"Missing the key {k!r}."
    assert d["seed"] == 1 and "numpy" in d["versions"]

test_file()
test_contents()
print("All checks passed")